# 02. Feature Exploration — 통계 검정 & ML Pipeline

`src/stats.py`, `src/pipeline.py`의 함수를 재사용해서 기술통계, 상관관계, t-test,
그리고 income 분류 모델 학습/평가까지 진행한다.

In [1]:
import sys
sys.path.insert(0, '..')

from src.clean import load_and_clean
from src.stats import descriptive_stats, correlation_matrix, income_ttest
from src.pipeline import train_and_evaluate

cleaned_df = load_and_clean()
cleaned_df.shape

중복 제거 전: 32,561행 -> 제거 후: 32,537행 (제거됨: 24행)


(32537, 15)

## 1. 기술통계

In [2]:
descriptive_stats(cleaned_df)

,count,mean,std,min,25%,50%,75%,max
age,32537.0,38.585549,13.637984,17.0,28.0,37.0,48.0,90.0
fnlwgt,32537.0,189780.848511,105556.471009,12285.0,117827.0,178356.0,236993.0,1484705.0
education-num,32537.0,10.081815,2.571633,1.0,9.0,10.0,12.0,16.0
capital-gain,32537.0,1078.443741,7387.957424,0.0,0.0,0.0,0.0,99999.0
capital-loss,32537.0,87.368227,403.101833,0.0,0.0,0.0,0.0,4356.0
hours-per-week,32537.0,40.440329,12.346889,1.0,40.0,40.0,45.0,99.0


## 2. 상관계수 행렬

In [3]:
correlation_matrix(cleaned_df)

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
age,1.000000,-0.076447,0.036224,0.077676,0.057745,0.068515
fnlwgt,-0.076447,1.000000,-0.043388,0.000429,-0.010260,-0.018898
education-num,0.036224,-0.043388,1.000000,0.122664,0.079892,0.148422
capital-gain,0.077676,0.000429,0.122664,1.000000,-0.031639,0.078408
capital-loss,0.057745,-0.010260,0.079892,-0.031639,1.000000,0.054229
hours-per-week,0.068515,-0.018898,0.148422,0.078408,0.054229,1.000000


## 3. t-test: income 그룹 간 주당 근무시간 차이

`>50K` 그룹과 `<=50K` 그룹의 `hours-per-week` 평균이 통계적으로 다른지 검정한다.

In [4]:
ttest_result = income_ttest(cleaned_df)
ttest_result

[t-test] '>50K' vs '<=50K' on 'hours-per-week'
  >50K 평균 : 45.47  (n=7,839)
  <=50K 평균 : 38.84  (n=24,698)
  t-statistic : 45.0950
  p-value     : 0.0000
  -> 유의수준 0.05에서 '>50K' 그룹과 '<=50K' 그룹의 'hours-per-week' 평균 차이는 통계적으로 유의하다.


{'value_col': 'hours-per-week',
 'group_col': 'income',
 'group_a': '>50K',
 'group_b': '<=50K',
 'mean_a': np.float64(45.473402219670874),
 'mean_b': np.float64(38.84286177018382),
 'n_a': 7839,
 'n_b': 24698,
 't_stat': np.float64(45.0950261530006),
 'p_value': np.float64(0.0),
 'alpha': 0.05,
 'is_significant': np.True_,
 'interpretation': "유의수준 0.05에서 '>50K' 그룹과 '<=50K' 그룹의 'hours-per-week' 평균 차이는 통계적으로 유의하다."}

## 4. ML Pipeline: income 분류 모델 학습/평가

`ColumnTransformer` + `LogisticRegression`으로 구성된 Pipeline을 학습하고,
`output/model.pkl`로 저장 후 재로딩까지 검증한다.

In [5]:
pipeline_result = train_and_evaluate(cleaned_df)
pipeline_result['accuracy'], pipeline_result['f1']

[Pipeline] train=26,029 / test=6,508
[Pipeline] accuracy=0.8569, f1=0.6762
              precision    recall  f1-score   support

       <=50K       0.89      0.93      0.91      4940
        >50K       0.74      0.62      0.68      1568

    accuracy                           0.86      6508
   macro avg       0.81      0.78      0.79      6508
weighted avg       0.85      0.86      0.85      6508

[Pipeline] 모델 저장 완료: /Users/hayeon/workspace/data-project/teamproject/output/model.pkl
[Pipeline] 재로딩 후 accuracy=0.8569 (원본과 일치: True)


(0.8569452980946527, 0.6761739130434783)